implementing gemini solutions

### Environment, GPU, and API Setup

In [1]:
import os
import torch
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Load Environment Variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# GPU Verification
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 Using GPU: {torch.cuda.get_device_name(0)}")

✅ CUDA Available: True
🚀 Using GPU: NVIDIA GeForce RTX 4060 Ti


In [2]:
# Use the fast model for the heavy-lifting research
search_llm = LLM(model="groq/llama-3.1-8b-instant", api_key=os.environ.get("GROQ_API_KEY"))

# Use the big model ONLY for the final answer
reasoning_llm = LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ.get("GROQ_API_KEY"))

### Vector Store Loading

In [3]:
# Initialize Embedding Model
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)

# Define Persistent Path (Update this to your actual folder path)
persist_directory = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\VectorDatabase2"

# Load the existing Vector Store
vector_store = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings,
    collection_name="legal_knowledge"
)

print(f"✅ Successfully loaded {vector_store._collection.count()} chunks from disk.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Successfully loaded 1796 chunks from disk.


### The Search Tool (With Instruction Guardrails)

In [18]:
# --- NEW LLM CONFIGURATION ---
# Use the 'instant' model for research—it has much higher rate limits
research_llm = LLM(model="groq/llama-3.1-8b-instant", api_key=os.environ.get("GROQ_API_KEY"))

# Use the '70b' ONLY for the final answer
legal_llm = LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ.get("GROQ_API_KEY"))

@tool("search_tool")
def legal_search_tool(query: str) -> str:
    """
    Search the Indian Constitution. 
    Input should be the USER SCENARIO or a specific legal question.
    """
    # 1. OPTIMIZATION: Extracting keywords to improve semantic matching
    # We do a 'small search' on the actual scenario and a 'keyword search'
    search_queries = [query]
    
    # Simple logic to add 'Indian Constitution' to the search to ground the embedding
    results = vector_store.similarity_search(f"Indian Constitution {query}", k=4)
    
    if not results:
        return "No relevant articles found. Please try searching for specific legal keywords like 'Right to Protest' or 'Self-Incrimination'."
        
    formatted_output = []
    for doc in results:
        # We use parent_context but keep it under the token limit for Groq
        context = doc.metadata.get('parent_context', doc.page_content)[:1200]
        label = doc.metadata.get('page_label', 'Constitution Section')
        formatted_output.append(f"ARTICLE/LABEL: {label}\nLEGAL TEXT: {context}")
        
    return "\n\n---\n\n".join(formatted_output)

### LLM and Agent Initialization

In [19]:
# --- UPDATED AGENTS ---
researcher = Agent(
    role="Legal Issue Spotter",
    goal="Identify the legal issues in the scenario and find the matching Indian Constitution Articles.",
    backstory=(
        "You are an expert at reading a story and identifying if it relates to "
        "Fundamental Rights, Directive Principles, or State Powers. You use the "
        "search_tool by searching for the core concepts (e.g., if a person is "
        "arrested, search for 'arrest' or 'confession')."
    ),
    tools=[legal_search_tool],
    llm=research_llm, 
    allow_delegation=False,
    verbose=True
)

legal_consultant = Agent(
    role="Senior Advocate of the Supreme Court of India",
    goal="Answer the legal query using ONLY the Indian Constitutional text provided by the researcher.",
    backstory=(
        "You are a legal expert in India. You interpret Article 20 and other "
        "relevant sections based strictly on the researcher's findings. "
        "If no text was found, do not make up an answer."
    ),
    llm=legal_llm, # Use the 70B model here for high-quality reasoning
    allow_delegation=False,
    verbose=True
)

### Sequential Task Definitions

In [20]:
# 4. STREAMLINED TASKS
research_task = Task(
    description="Search for all Articles related to: {user_query}. Provide the raw text.",
    expected_output="A collection of relevant Constitutional Articles.",
    agent=researcher
)

analysis_task = Task(
    description="Analyze the research and answer: {user_query}. Cite specific Articles.",
    expected_output="A structured legal response.",
    agent=legal_consultant,
    context=[research_task]
)

### Crew Execution (Rate Limit Guard)

In [21]:
# 5. RATE-LIMITED CREW
legal_crew = Crew(
    agents=[researcher, legal_consultant],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    max_rpm=1, # Strictly limit requests per minute for Groq
    cache=True
)

### Query

In [9]:
# Execute query
user_input = "Can a person be forced to provide a blood sample as evidence ?"

print(f"🚀 LegalRAG Chamber is analyzing: {user_input}\n")
# CRITICAL: Pass 'user_input' variable, NOT a placeholder string
result = legal_crew.kickoff(inputs={"user_query": user_input})

print(result.raw)

# Final Output Display
print("\n" + "="*60)
print("FINAL VERIFIED LEGAL OPINION")
print("="*60)
print(result.raw)

🚀 LegalRAG Chamber is analyzing: Can a person be forced to provide a blood sample as evidence ?



╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Indian Constitutional Researcher                                                                        │
│                                                                                                                 │
│  Task: Search for all Articles related to: Can a person be forced to provide a blood sample as evidence ?.      │
│  Provide the raw text.                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_tool executed with result: SOURCE: 4
TEXT: THE CONSTITUTION OF INDIA

---

SOURCE: 4
TEXT: THE CONSTITUTION OF INDIA

---

SOURCE: 54
TEXT: THE CONSTITUTION OF  INDIA
(Part IV. Directive Principles of State Policy)
23
1[43B. Pr...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Indian Constitutional Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The search_tool did not return the exact text of Article 20 of the Indian Constitution regarding the forced    │
│  provision of blood samples as evidence.                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Advocate of the Supreme Court of India                                                           │
│                                                                                                                 │
│  Task: Analyze the research and answer: Can a person be forced to provide a blood sample as evidence ?. Cite    │
│  specific Articles.                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Advocate of the Supreme Court of India                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **LEGAL RESPONSE**                                                                                             │
│                                                                                                                 │
│  To determine whether a person can be forced to provide a blood sample as evidence, it is essential to examine  │
│  the relevant provisions of the Indian Constitution and legal framework. Unfortunately, as the search tool did  │
│  not return the exact text of Article 20 of the Indian Constitution regarding the forced provision of blood     │
│  samples as evidence, I am constrained to provide a response based on general legal principles and the typical  │
│  content of relevant constitutional articles.                                                                   │
│                                                                                                                 │
│  **PROTECTION AGAINST SELF-INCRIMINATION**                                                                      │
│                                                                                                                 │
│  In the Indian legal system, the right against self-incrimination is a fundamental principle. This right is     │
│  generally understood to be protected under Article 20(3) of the Indian Constitution, which states that no      │
│  person accused of any offence shall be compelled to be a witness against himself. However, without the exact   │
│  text of Article 20 provided, I must rely on this general understanding.                                        │
│                                                                                                                 │
│  The concept of self-incrimination encompasses not only verbal testimony but also other forms of evidence that  │
│  an individual might be compelled to provide, which could lead to their incrimination. The question of whether  │
│  providing a blood sample constitutes self-incrimination depends on the context and the potential uses of such  │
│  evidence.                                                                                                      │
│                                                                                                                 │
│  **RELEVANT LEGAL FRAMEWORK**                                                                                   │
│                                                                                                                 │
│  In the absence of specific guidance from Article 20, other legal provisions and judicial precedents might      │
│  offer insight. For instance, the Code of Criminal Procedure, 1973, and the Indian Evidence Act, 1872, contain  │
│  provisions related to the collection and admissibility of evidence in criminal proceedings. These laws         │
│  outline the procedures for obtaining various types of evidence, including medical evidence, but the specifics  │
│  regarding the compulsory provision of blood samples would require a detailed examination of these statutes     │
│  and relevant case law.                                                                                         │
│                                                                                                                 │
│  **CONCLUSION**                                        

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

**LEGAL RESPONSE**

To determine whether a person can be forced to provide a blood sample as evidence, it is essential to examine the relevant provisions of the Indian Constitution and legal framework. Unfortunately, as the search tool did not return the exact text of Article 20 of the Indian Constitution regarding the forced provision of blood samples as evidence, I am constrained to provide a response based on general legal principles and the typical content of relevant constitutional articles.

**PROTECTION AGAINST SELF-INCRIMINATION**

In the Indian legal system, the right against self-incrimination is a fundamental principle. This right is generally understood to be protected under Article 20(3) of the Indian Constitution, which states that no person accused of any offence shall be compelled to be a witness against himself. However, without the exact text of Article 20 provided, I must rely on this general understanding.

The concept of self-incrimination encompasses not only verb

In [22]:
import pandas as pd
import json
import time

# 1. Situational Test Dataset (Updated with your 5 specific scenarios)
test_dataset = [
    {
        "question": "A group of residents is protesting peacefully on a public sidewalk against a new local law. The police arrive and tell them they must disperse immediately because they don't like the 'message' of the protest. Is this dispersal constitutional?",
        "expected_article": "Article 19(1)(b) and 19(2)",
        "ground_truth": "No. Under Article 19(1)(b), citizens have the right to assemble peaceably and without arms. Restrictions under 19(2) can only be imposed for specific reasons like 'public order'. Dispersing based on 'message' is unconstitutional."
    },
    {
        "question": "A state government passes a law mandating that all primary education in private schools must be conducted only in the state's regional language, prohibiting English. Do the schools have any constitutional protection?",
        "expected_article": "Article 30 and Article 19(1)(g)",
        "ground_truth": "Yes. Article 30 grants minorities the right to establish and administer educational institutions. Article 19(1)(g) protects the right to practice a profession. Linguistic mandates cannot override administrative autonomy."
    },
    {
        "question": "The government decides to build a highway through a historic religious shrine. The religious community claims this violates their right to manage their own religious affairs. Who is right according to the law?",
        "expected_article": "Article 26 vs Article 25",
        "ground_truth": "Article 26 protects management of religious affairs, but the state can regulate secular activities. Land acquisition for public infrastructure usually prevails if it doesn't destroy essential religious practice."
    },
    {
        "question": "A person is arrested for a crime. During the trial, the prosecution wants to use a confession the person made while they were in police custody, even though the person now claims they were threatened. Can this confession be used?",
        "expected_article": "Article 20(3)",
        "ground_truth": "No. Article 20(3) provides that no person shall be compelled to be a witness against himself. Confessions under threat or in police custody are generally inadmissible."
    },
    {
        "question": "An employer refuses to hire a qualified candidate because they belong to a specific caste, claiming it's a private business decision. Does the candidate have a constitutional remedy?",
        "expected_article": "Article 15 and Article 16",
        "ground_truth": "Yes. Article 15(2) prohibits discrimination in access to public places, and the spirit of the Constitution (Article 17 and civil rights laws) prohibits caste-based discrimination in the public sphere."
    }
]

results_log = []
# Using 70b as the judge for higher reasoning quality
judge_llm = LLM(model="groq/llama-3.3-70b-versatile", api_key=os.environ.get("GROQ_API_KEY"))

print(f"🚀 Starting Situational Evaluation for {len(test_dataset)} cases...\n")

for case in test_dataset:
    print(f"--- Testing Scenario: {case['question'][:50]}... ---")
    
    try:
        # 1. Run the Agentic Crew
        # The 'max_rpm=1' in your Crew definition is critical here
        output = legal_crew.kickoff(inputs={"user_query": case['question']})
        actual_answer = output.raw
        
        # 2. Advanced Grading Prompt
        grade_prompt = f"""
        You are a Senior Constitutional Judge. Evaluate the AI's legal advice.
        
        USER SCENARIO: {case['question']}
        EXPECTED BASIS: {case['expected_article']}
        GROUND TRUTH: {case['ground_truth']}
        AI RESPONSE: {actual_answer}
        
        Rate from 1-5:
        1. ACCURACY: Correct Article identification?
        2. REASONING: Correct application to the scenario?
        3. FAITHFULNESS: Avoided foreign (US) law?
        
        Return ONLY JSON: {{"accuracy": score, "reasoning": score, "faithfulness": score, "feedback": "short feedback"}}
        """
        
        # 3. Call Judge with a small safety delay
        time.sleep(2) 
        response = judge_llm.call([{"role": "user", "content": grade_prompt}])
        
        # Clean and parse JSON
        clean_json = response.strip().replace("```json", "").replace("```", "")
        evaluation = json.loads(clean_json)
        
        results_log.append({
            "Scenario": case['question'][:60] + "...",
            "Score (Acc/Reas/Faith)": f"{evaluation['accuracy']}/{evaluation['reasoning']}/{evaluation['faithfulness']}",
            "Judge Feedback": evaluation['feedback']
        })
        
        # 4. Mandatory Wait to reset the 6000 TPM limit of the 8b model used in the Crew
        print("Waiting 15 seconds to respect Groq TPM limits...")
        time.sleep(15)

    except Exception as e:
        error_msg = str(e)
        print(f"⚠️ Error grading case: {error_msg[:100]}")
        results_log.append({
            "Scenario": case['question'][:60] + "...",
            "Score (Acc/Reas/Faith)": "ERROR",
            "Judge Feedback": f"System failed: {error_msg[:50]}"
        })
        time.sleep(20) # Longer wait after an error

# Display Results
df_results = pd.DataFrame(results_log)
print("\n✅ Situational Evaluation Complete!")
display(df_results)

🚀 Starting Situational Evaluation for 5 cases...

--- Testing Scenario: A group of residents is protesting peacefully on a... ---


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Legal Issue Spotter                                                                                     │
│                                                                                                                 │
│  Task: Search for all Articles related to: A group of residents is protesting peacefully on a public sidewalk   │
│  against a new local law. The police arrive and tell them they must disperse immediately because they don't     │
│  like the 'message' of the protest. Is this dispersal constitutional?. Provide the raw text.                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_tool executed with result: ARTICLE/LABEL: 4
LEGAL TEXT: THE CONSTITUTION OF INDIA

---

ARTICLE/LABEL: 4
LEGAL TEXT: THE CONSTITUTION OF INDIA

---

ARTICLE/LABEL: 40
LEGAL TEXT: THE CONSTITUTION OF  INDIA
(Part III.Fundamental...


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_error' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_error' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

⚠️ Error grading case: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for 

✅ Situational Evaluation Complete!


,Scenario,Score (Acc/Reas/Faith),Judge Feedback
0,A group of residents is protesting peacefully ...,ERROR,System failed: litellm.BadRequestError: GroqEx...
1,A state government passes a law mandating that...,ERROR,System failed: litellm.BadRequestError: GroqEx...
2,The government decides to build a highway thro...,ERROR,System failed: litellm.BadRequestError: GroqEx...
3,A person is arrested for a crime. During the t...,ERROR,System failed: litellm.RateLimitError: RateLim...
4,An employer refuses to hire a qualified candid...,ERROR,System failed: litellm.RateLimitError: RateLim...
